# Day 15 — Anomaly Baseline + Mentor Review

**Goal:** compare a simple statistical anomaly rule with an Isolation Forest detector and visualize candidate anomalies.

> **Critical data limitation:** The source is a static water-potability table with no genuine timestamp. The timestamp column was generated synthetically. These experiments demonstrate an anomaly-detection workflow only; flagged rows are *candidates*, not verified environmental events or sensor faults. Do not claim real-time temporal anomaly performance from this dataset.

## Methods
1. **Statistical baseline:** learn Q1/Q3 and IQR bounds from the earliest 70% of rows only. A row is flagged if any sensor value falls outside the training bounds.
2. **Isolation Forest:** median imputation, scaling, and Isolation Forest are fitted using the earliest 70% chronological rows only. The `contamination=0.03` value is a working setting, not a known true anomaly rate.
3. **Review:** compare flagged rows and inspect plots. No ground-truth anomaly labels exist, so precision/recall cannot be honestly calculated.

`Potability` is not used as an anomaly label. It is a separate dataset target whose meaning should be confirmed with the mentor.

In [ ]:
# Imports and data loading
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
from sklearn.pipeline import Pipeline

# If running in the same folder as the extracted source CSV, use that path.
# Otherwise update DATA_PATH to the synthetic-timestamp CSV in your project.
DATA_PATH = "../underwater_temporal_dataset/water_potability_temporal_synthetic.csv"
df = pd.read_csv(DATA_PATH)
df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")
sensor_cols = ["ph", "Hardness", "Solids", "Chloramines", "Sulfate",
               "Conductivity", "Organic_carbon", "Trihalomethanes", "Turbidity"]
df = df.sort_values("timestamp", kind="stable").reset_index(drop=True)
print("Shape:", df.shape)
print("Rows with missing timestamps:", df["timestamp"].isna().sum())
print("Missing values by sensor:\n", df[sensor_cols].isna().sum())

## 1. Chronological split and statistical IQR baseline

The earliest 70% of rows are used to estimate thresholds. The remaining rows are later observations. The split is for a reproducible review workflow, not proof that synthetic ordering represents real time.

In [ ]:
split_idx = int(len(df) * 0.70)
X = df[sensor_cols].copy()
X_train = X.iloc[:split_idx].copy()

q1 = X_train.quantile(0.25)
q3 = X_train.quantile(0.75)
iqr = q3 - q1
lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr

outside = X.lt(lower, axis=1) | X.gt(upper, axis=1)
df["iqr_flagged_sensor_count"] = outside.sum(axis=1)
df["iqr_anomaly"] = outside.any(axis=1)

print("IQR bounds learned from training rows only:")
display(pd.DataFrame({"lower": lower, "upper": upper}))
print("IQR flagged rows:", int(df["iqr_anomaly"].sum()))

## 2. Isolation Forest

Missing values are imputed using training medians, and scaling is fitted only on training rows. Isolation Forest is also fitted only on training rows, then applied to the complete chronological series. The contamination setting is a modeling assumption and should be reviewed with the mentor.

In [ ]:
model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("iforest", IsolationForest(
        n_estimators=200,
        contamination=0.03,  # working assumption; tune only with a defensible protocol
        random_state=42
    ))
])
model.fit(X_train)
df["iforest_prediction"] = model.predict(X)  # -1 = candidate anomaly, +1 = inlier
df["iforest_score"] = model.decision_function(X)
df["iforest_anomaly"] = df["iforest_prediction"].eq(-1)

print("Isolation Forest flagged rows:", int(df["iforest_anomaly"].sum()))
print("IQR flagged rows:", int(df["iqr_anomaly"].sum()))
print("Flagged by both:", int((df["iforest_anomaly"] & df["iqr_anomaly"]).sum()))

## 3. Visualize candidate anomalies

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
for ax, col in zip(axes, ["ph", "Turbidity"]):
    ax.plot(df["timestamp"], df[col], linewidth=0.8, label=col)
    mask = df["iforest_anomaly"] & df[col].notna()
    ax.scatter(df.loc[mask, "timestamp"], df.loc[mask, col],
               s=18, label="Isolation Forest flagged", zorder=3)
    ax.set_ylabel(col)
    ax.legend()
    ax.grid(True, alpha=0.25)
axes[-1].set_xlabel("Synthetic timestamp")
fig.suptitle("Candidate anomalies over synthetic time — demonstration only")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(9, 5))
plt.hist(df["iforest_score"].dropna(), bins=40)
plt.title("Isolation Forest decision-score distribution")
plt.xlabel("Decision function score (lower = more anomalous)")
plt.ylabel("Rows")
plt.grid(True, alpha=0.25)
plt.show()

In [ ]:
counts = pd.Series({
    "IQR statistical baseline": int(df["iqr_anomaly"].sum()),
    "Isolation Forest": int(df["iforest_anomaly"].sum()),
    "Both methods": int((df["iqr_anomaly"] & df["iforest_anomaly"]).sum())
})
display(counts.to_frame("flagged_rows"))
counts.plot(kind="bar", figsize=(8, 5), title="Candidate anomaly counts by method",
            ylabel="Flagged rows", rot=12)
plt.tight_layout()
plt.show()

## 4. Export and review candidate rows

Inspect the flagged rows against sensor context and any domain limits provided by the mentor. Do not automatically delete or replace flagged values. An unusual value may be valid. The dataset has no verified anomaly labels, so use agreement between methods only as a review prioritization aid.

In [ ]:
result_cols = ["timestamp"] + sensor_cols
if "Potability" in df.columns:
    result_cols += ["Potability"]
result_cols += ["iqr_flagged_sensor_count", "iqr_anomaly",
                "iforest_anomaly", "iforest_score"]
candidate_rows = df.loc[df["iqr_anomaly"] | df["iforest_anomaly"], result_cols].copy()
candidate_rows.to_csv("anomaly_detection_candidates.csv", index=False)
display(candidate_rows.head(20))
print("Saved anomaly_detection_candidates.csv")

## 5. Evaluation strategy to confirm with mentor

Because there are no ground-truth anomaly labels:
- **Do not report classification accuracy, precision, recall, or F1 for anomalies.**
- Report number and percentage flagged by each method, overlap between methods, sensor-level flags, and a reviewed sample of candidates.
- If the mentor can provide validated anomaly labels or sensor operating limits, evaluate against those labels/limits.
- Fit imputation, scaling, thresholds, and detector only on training data; do not use later rows to tune thresholds.
- Confirm the chronological split, IQR multiplier, Isolation Forest contamination, and the process for validating candidate anomalies before freezing the baseline.

## Preliminary conclusion
The methods are unsupervised screening baselines. Results are pipeline outputs, not proof of faulty sensors. Since the timestamps are synthetic, temporal clustering and trend interpretations must be treated as artificial until genuine timestamped environmental data is available.